# NumPy for AI/ML

Why are we even talking about NumPy in an AI/ML bootcamp?

Every model you'll ever train from linear regression to deep neural networks is doing math on arrays of numbers under the hood. NumPy is the  library that makes that math fast and clean in Python. Scikit-learn, Pandas, TensorFlow, PyTorch  they're all built on top of it.

Today's roadmap:
1. Why NumPy? (the speed problem)
2. The `ndarray` NumPy's core object
3. Creating arrays fast
4. Indexing & slicing
5. Reshaping
6. Broadcasting (the big one)
7. Vectorized math & linear algebra
8. Live demo: normalizing a dataset
9. Wrap-up


 Why NumPy? 

Let's do the same task two ways: square every number in a list of a million numbers.

- Way 1: plain Python for loop
- Way 2: NumPy array operation




In [ ]:
import numpy as np
import time

n = 1_000_000

# Way 1: pure Python list + loop
python_list = list(range(n))

start = time.time()
squared_list = [x**2 for x in python_list]
python_time = time.time() - start

# Way 2: NumPy array
numpy_array = np.arange(n)

start = time.time()
squared_array = numpy_array ** 2
numpy_time = time.time() - start

print(f"Python loop time: {python_time:.5f} sec")
print(f"NumPy time:       {numpy_time:.5f} sec")
print(f"NumPy is roughly {python_time / numpy_time:.0f}x faster")


Python loop time: 0.08873 sec
NumPy time:       0.00408 sec
NumPy is roughly 22x faster


Takeaway: NumPy isn't just "a library for arrays." It's written in C under the hood, so operations run at compiled-language speed instead of Python's interpreted speed. When your dataset has millions of rows or your neural network has millions of weights, this difference is the difference between training in seconds vs. hours.


## 2. The ndarray NumPy's Core Object

Everything in NumPy revolves around the n-dimensional array (ndarray). Think of it as a Python list's much faster, much stricter cousin — all elements must be the same data type, and it knows its own shape.


In [ ]:
# Creating arrays from lists
vector = np.array([1, 2, 3, 4])           # 1D array
matrix = np.array([[1, 2, 3], [4, 5, 6]]) # 2D array

print("Vector:", vector)
print("Matrix:\n", matrix)

print("\n--- Key attributes ---")
print("Shape of vector:", vector.shape)
print("Shape of matrix:", matrix.shape)   # (rows, columns)
print("Dimensions (ndim):", matrix.ndim)
print("Data type (dtype):", matrix.dtype)
print("Total elements (size):", matrix.size)


Vector: [1 2 3 4]
Matrix:
 [[1 2 3]
 [4 5 6]]

--- Key attributes ---
Shape of vector: (4,)
Shape of matrix: (2, 3)
Dimensions (ndim): 2
Data type (dtype): int64
Total elements (size): 6


**Why this matters for ML:** every dataset you load, every image, every batch of training data becomes an ndarray with a specific shape. 


## 3. Creating Arrays Fast

In real ML code, you rarely type out arrays by hand  you generate them.


In [ ]:
zeros = np.zeros((3, 3))          # useful for initializing weight matrices
ones = np.ones((2, 4))             # useful for masks/bias terms
identity = np.eye(3)               # identity matrix - common in linear algebra
range_arr = np.arange(0, 10, 2)    # like Python's range(), but returns an array
linspace_arr = np.linspace(0, 1, 5)  # 5 evenly spaced values between 0 and 1

print("Zeros:\n", zeros)
print("\nOnes:\n", ones)
print("\nIdentity:\n", identity)
print("\nArange:", range_arr)
print("\nLinspace:", linspace_arr)

# Random arrays - used to initialize model weights before training
np.random.seed(42)  # for reproducibility
random_arr = np.random.rand(2, 3)       # uniform [0, 1)
random_normal = np.random.randn(2, 3)   # standard normal distribution
print("\nRandom uniform:\n", random_arr)
print("\nRandom normal:\n", random_normal)


Zeros:
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

Ones:
 [[1. 1. 1. 1.]
 [1. 1. 1. 1.]]

Identity:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Arange: [0 2 4 6 8]

Linspace: [0.   0.25 0.5  0.75 1.  ]

Random uniform:
 [[0.37454012 0.95071431 0.73199394]
 [0.59865848 0.15601864 0.15599452]]

Random normal:
 [[ 1.57921282  0.76743473 -0.46947439]
 [ 0.54256004 -0.46341769 -0.46572975]]


Why this matters for ML: when a neural network is built, its weights don't start at "nothing" they start as small random numbers, usually generated with `np.random.randn()`. This is literally step one of training any model.


## 4. Indexing & Slicing

Same idea as Python lists, but more powerful  especially boolean masking, which you'll use constantly for filtering data.


In [ ]:
data = np.array([10, 20, 30, 40, 50, 60])

print("Single element:", data[2])
print("Slice [1:4]:", data[1:4])
print("Last 2 elements:", data[-2:])

# 2D indexing
matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("\nElement at row 1, col 2:", matrix[1, 2])
print("Entire row 0:", matrix[0, :])
print("Entire column 1:", matrix[:, 1])

# Boolean masking : THE most used indexing trick in ML preprocessing
print("\n Boolean masking ")
mask = data > 30
print("Mask:", mask)
print("Filtered values (>30):", data[mask])

# One-liner version, very common in real code
print("Values > 30, one line:", data[data > 30])


Single element: 30
Slice [1:4]: [20 30 40]
Last 2 elements: [50 60]

Element at row 1, col 2: 6
Entire row 0: [1 2 3]
Entire column 1: [2 5 8]

--- Boolean masking ---
Mask: [False False False  True  True  True]
Filtered values (>30): [40 50 60]
Values > 30, one line: [40 50 60]


**Why this matters for ML:** cleaning data remove all rows where age is negative,keep only rows where the label is 1  is almost always written as a boolean mask like data[data > 30].


# 5. Reshaping

Data doesn't always arrive in the shape your model needs. Reshaping lets you rearrange the same values into a different structure.


In [ ]:
arr = np.arange(12)
print("Original:", arr, "shape:", arr.shape)

reshaped = arr.reshape(3, 4)
print("\nReshaped to (3, 4):\n", reshaped)

reshaped_again = arr.reshape(2, 2, 3)
print("\nReshaped to (2, 2, 3):\n", reshaped_again)

flattened = reshaped.flatten()
print("\nFlattened back to 1D:", flattened)


auto_reshape = arr.reshape(4, -1)
print("\nAuto-reshaped (4, -1):\n", auto_reshape)


Original: [ 0  1  2  3  4  5  6  7  8  9 10 11] shape: (12,)

Reshaped to (3, 4):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Reshaped to (2, 2, 3):
 [[[ 0  1  2]
  [ 3  4  5]]

 [[ 6  7  8]
  [ 9 10 11]]]

Flattened back to 1D: [ 0  1  2  3  4  5  6  7  8  9 10 11]

Auto-reshaped (4, -1):
 [[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


Why this matters for ML: a classic example is image data. A 28×28 grayscale image is a (28, 28) array, but many models expect a flat vector — so you reshape(784) or flatten() it before feeding it in.


# 6. Broadcasting  The Most Important Concept Today

Broadcasting is how NumPy performs operations on arrays of **different shapes** without you writing a loop. It "stretches" the smaller array to match the bigger one, following a set of rules — no data is actually copied, it's just implied.

This is the single concept that makes vectorized ML code possible.


In [ ]:
# Scalar broadcasting 
arr = np.array([1, 2, 3])
print("Array + scalar:", arr + 10)

# Vector broadcast across a matrix
matrix = np.array([[1, 2, 3],
                    [4, 5, 6],
                    [7, 8, 9]])
row_vector = np.array([10, 20, 30])

print("\nMatrix:\n", matrix)
print("\nMatrix + row_vector (broadcast across each row):\n", matrix + row_vector)

# This is EXACTLY how bias is added in a neural network layer:



Array + scalar: [11 12 13]

Matrix:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]

Matrix + row_vector (broadcast across each row):
 [[11 22 33]
 [14 25 36]
 [17 28 39]]


Why this matters for ML: in a neural network layer, you compute `inputs @ weights + bias`. The `bias` is a single vector, but it gets added to *every row* of your batch — that's broadcasting in action. Without it, you'd need a manual loop over every sample.


## 7. Vectorized Math & Linear Algebra Basics

These are the actual operations that power ML algorithms.


In [ ]:
data = np.array([[1, 2, 3],
                  [4, 5, 6]])

print("Sum of all elements:", data.sum())
print("Sum along columns (axis=0):", data.sum(axis=0))
print("Sum along rows (axis=1):", data.sum(axis=1))
print("Mean:", data.mean())
print("Standard deviation:", data.std())

# Dot product - the core operation in every neural network layer
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print("\nDot product of two vectors:", np.dot(a, b))

# Matrix multiplication  "inputs times weights"
inputs = np.array([[1, 2], [3, 4]])   
weights = np.array([[0.5, 0.1], [0.2, 0.3]])  
output = inputs @ weights   # @ is matrix multiplication in NumPy
print("\nInputs:\n", inputs)
print("Weights:\n", weights)
print("Output (inputs @ weights):\n", output)


Sum of all elements: 21
Sum along columns (axis=0): [5 7 9]
Sum along rows (axis=1): [ 6 15]
Mean: 3.5
Standard deviation: 1.707825127659933

Dot product of two vectors: 32

Inputs:
 [[1 2]
 [3 4]]
Weights:
 [[0.5 0.1]
 [0.2 0.3]]
Output (inputs @ weights):
 [[0.9 0.7]
 [2.3 1.5]]


Why this matters for ML:inputs @ weights is *literally* what happens inside every layer of a neural network.


## 8. Live Demo: Normalizing a Dataset

A very common preprocessing step before training any model: **standardization** — rescale each feature to have mean 0 and standard deviation 1. Models train faster and more reliably on standardized data.

Formula: `standardized = (value - mean) / std`


In [ ]:
# Fake dataset: 5 samples, 3 features (e.g. height, weight, age)
dataset = np.array([
    [170, 65, 25],
    [160, 55, 30],
    [180, 80, 22],
    [175, 70, 28],
    [165, 60, 35]
], dtype=float)

print("Original dataset:\n", dataset)

# Compute mean and std PER COLUMN (axis=0 = down each feature)
column_means = dataset.mean(axis=0)
column_stds = dataset.std(axis=0)

print("\nColumn means:", column_means)
print("Column stds:", column_stds)

# Broadcasting does all the work here - no loop needed!
standardized = (dataset - column_means) / column_stds

print("\nStandardized dataset:\n", standardized)
print("\nCheck - new means (should be ~0):", standardized.mean(axis=0).round(2))
print("Check - new stds (should be ~1):", standardized.std(axis=0).round(2))


Original dataset:
 [[170.  65.  25.]
 [160.  55.  30.]
 [180.  80.  22.]
 [175.  70.  28.]
 [165.  60.  35.]]

Column means: [170.  66.  28.]
Column stds: [7.07106781 8.60232527 4.42718872]

Standardized dataset:
 [[ 0.         -0.11624764 -0.67763093]
 [-1.41421356 -1.27872403  0.45175395]
 [ 1.41421356  1.62746694 -1.35526185]
 [ 0.70710678  0.46499055  0.        ]
 [-0.70710678 -0.69748583  1.58113883]]

Check - new means (should be ~0): [ 0. -0.  0.]
Check - new stds (should be ~1): [1. 1. 1.]


**What just happened:** `dataset - column_means` subtracts a 3-element vector from every row of a 5x3 matrix — that's broadcasting. This exact pattern is what `StandardScaler` in Scikit-learn does internally.


# 9. Wrap-Up — 3 Things to Remember

1. NumPy is fast because it's vectorized — write operations on whole arrays, not loops over elements.
2. Broadcasting lets different-shaped arrays work together automatically — it's the trick behind adding bias terms, normalizing data, and more.
3. Matrix multiplication (@) is the mathematical core of most ML models — from linear regression to neural networks.

Thanks for listening!
